# Phase 3 - Rhetorical Role Prediction (hybrid RNN + transformer Phase-1 model)

**Task:** label every sentence of a legal document with one of 13 rhetorical roles (CL and IT domains); scored by macro-F1 over the 13 labels.

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 hybrid checkpoint `hybrid_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach:** consecutive sentences of a document are packed into one window (token budget `MAX_TOK`) and run through the hybrid (GRU blocks + attention blocks); each sentence is pooled into a vector (mean + last-token state), and a BiGRU over the window's sentences adds left *and* right sentence context before a 13-way classifier. `mode="sent"` removes all context for the context ablation.

**Experiments:** full fine-tuning, LoRA, rsLoRA, LoRMA+, IA3, prompt tuning, knowledge distillation (student = hybrid with one GRU block dropped per group), plus a sentence-only ablation. QLoRA, AdaLoRA and prefix tuning are **not** included.

> The course requires all **four** Phase-1 architectures and >= 6 substantially different experiments, at least one being knowledge distillation. Hyperparameters are placeholders - choosing and justifying them is your job (course rules).

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/hybrid_final.pt"      # <-- Phase-1 hybrid checkpoint
HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT = "/kaggle/working"
MAX_TOK  = 512      # token budget per window (<= the hybrid's max_seq_len of 1024)
SENT_TOK = 96       # each sentence is truncated to this many tokens
CTX_LEFT = 4        # up to this many preceding sentences are added as read-only context for each window

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 hybrid code (copied unchanged) and checkpoint loading
Embedding the class definitions is what lets the notebook work with only the `.pt` file.

In [ ]:
# ---- attention_ed.py pieces used by the hybrid (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x_f = x.float()
        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x_f * self.weight).to(x.dtype)


def build_rope_cache(max_len: int, d_head: int, base: float = 10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    angles = torch.outer(torch.arange(max_len, dtype=torch.float32), inv_freq)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin):
    x_f = x.float()
    x1, x2 = x_f[..., 0::2], x_f[..., 1::2]
    cos, sin = cos[None, None, :, :], sin[None, None, :, :]
    out = torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)
    return out.to(x.dtype)


# ---- hybrid_layers.py (Phase 1, unchanged) ----
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)


class FastRecurrentBlock(nn.Module):
    """Pre-norm cuDNN GRU block (unidirectional) + FFN."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.rnn = nn.GRU(input_size=d_model, hidden_size=d_model, num_layers=1, batch_first=True)
        self.ln2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
    def forward(self, x):
        r, _ = self.rnn(self.ln1(x))
        x = x + r
        return x + self.ffn(self.ln2(x))


class FastAttentionBlock(nn.Module):
    """Causal transformer block with RoPE + QK-norm, fused scaled_dot_product_attention."""
    def __init__(self, d_model, n_heads, max_seq_len, d_ff, dropout, rope=True, qk_norm=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_head = d_model, n_heads, d_model // n_heads
        self.ln1 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.rope = rope
        if rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.ln2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = dropout

    def apply_rope(self, x):
        return apply_rope(x, self.rope_cos[:x.shape[2]], self.rope_sin[:x.shape[2]])

    def forward(self, x):
        B, T, C = x.shape
        r = self.ln1(x)
        q = self.q_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rope:
            q, k = self.apply_rope(q), self.apply_rope(k)
        r = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                           dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        r = self.out_proj(r.transpose(1, 2).contiguous().view(B, T, C))
        x = x + r
        return x + self.ffn(self.ln2(x))


# ---- model_arch_hybrid.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class HybridConfig:
    vocab_size: int
    d_model: int = 768
    n_heads: int = 12
    n_recurrent_layers: int = 8
    n_attention_layers: int = 4
    d_ff: int = 2304
    max_seq_len: int = 1024
    dropout: float = 0.0
    rope: bool = True
    qk_norm: bool = True


class HybridLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        blocks = []
        for _ in range(4):                                   # [R R A] x 4
            blocks.append(FastRecurrentBlock(cfg.d_model, cfg.d_ff, cfg.dropout))
            blocks.append(FastRecurrentBlock(cfg.d_model, cfg.d_ff, cfg.dropout))
            blocks.append(FastAttentionBlock(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.d_ff, cfg.dropout,
                                             rope=cfg.rope, qk_norm=cfg.qk_norm))
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))

In [ ]:
def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = HybridConfig(**{k: v for k, v in ck["config"].items() if k in HybridConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = HybridLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # rope caches are non-persistent buffers: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 hybrid: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M params, {len(_lm.blocks)} blocks")
assert _lm.cfg.max_seq_len >= MAX_TOK + 64, "MAX_TOK (+ prompt tokens) must fit in max_seq_len"

## 4. Data
### 4.1 Peek at the raw JSON (verify the layout before trusting the parser)

In [ ]:
def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

lv = hf_json("Phase_3_RR/Train/label_vocab.json")
print("label_vocab:", type(lv).__name__, str(lv)[:600], "\n")
for fn in ["CL_train", "IT_train", "CL_test"]:
    raw = hf_json(f"Phase_3_RR/{'Test' if fn.endswith('test') else 'Train'}/{fn}.json")
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    print(fn, type(raw).__name__, "len", len(raw), "\n   first:", str(first)[:500], "\n")

### 4.2 Parse CL + IT train / dev / test, check integrity

In [ ]:
# ---- labels (exactly as in label_vocab.json) ----
if isinstance(lv, dict):
    if all(isinstance(v, int) for v in lv.values()):
        labels = [k for k, _ in sorted(lv.items(), key=lambda kv: kv[1])]
    else:
        labels = [lv[k] for k in sorted(lv, key=lambda x: int(x))]
else:
    labels = list(lv)
label2id = {l: i for i, l in enumerate(labels)}
N_LAB = len(labels)
print(N_LAB, "labels:", labels)

def parse_docs(raw, dom):
    items = raw.items() if isinstance(raw, dict) else ((d["id"], d) for d in raw)
    docs = []
    for did, d in items:
        docs.append(dict(id=str(did), dom=dom, key=f"{dom}|{did}", sents=list(d["text"]), labels=d.get("labels")))
    return docs

def load_split(dom, split):
    sub = "Test" if split == "test" else "Train"
    return parse_docs(hf_json(f"Phase_3_RR/{sub}/{dom}_{split}.json"), dom)

train_docs = load_split("CL", "train") + load_split("IT", "train")
dev_docs   = load_split("CL", "dev")   + load_split("IT", "dev")
test_docs  = load_split("CL", "test")  + load_split("IT", "test")
for name, ds in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    n_s = sum(len(d["sents"]) for d in ds)
    print(f"{name}: {len(ds)} docs, {n_s} sentences (CL {sum(d['dom']=='CL' for d in ds)} docs / IT {sum(d['dom']=='IT' for d in ds)} docs)")

# integrity checks
for d in train_docs + dev_docs:
    assert len(d["sents"]) == len(d["labels"]), f"len(text) != len(labels) in {d['key']}"
    bad = set(d["labels"]) - set(labels)
    assert not bad, f"labels not in vocab: {bad}"
assert len({d["key"] for d in test_docs}) == len(test_docs), "duplicate test keys"
ids_cl = {d["id"] for d in test_docs if d["dom"] == "CL"}; ids_it = {d["id"] for d in test_docs if d["dom"] == "IT"}
print("test id overlap between CL and IT:", len(ids_cl & ids_it), "(must be 0 if you submit one merged prediction.json)")

from collections import Counter
cnt = Counter(l for d in train_docs for l in d["labels"])
print({l: cnt.get(l, 0) for l in labels})

### 4.3 Tokenise sentences

In [ ]:
def enc_sentence(s):
    ids = tokenizer.encode(" " + s)[:SENT_TOK]
    return ids if ids else [tokenizer.eos_id]

t0 = time.time()
for d in train_docs + dev_docs + test_docs:
    d["ids"] = [enc_sentence(s) for s in d["sents"]]
lens = [len(i) for d in train_docs for i in d["ids"]]
print(f"tokenised in {time.time()-t0:.0f}s | tokens/sentence: median {int(np.median(lens))}, "
      f"p95 {int(np.percentile(lens, 95))}, max {max(lens)} | sentences/doc: "
      f"median {int(np.median([len(d['ids']) for d in train_docs]))}, max {max(len(d['ids']) for d in train_docs)}")

### 4.4 Windows (sentence packing with left context) and batching

In [ ]:
def make_windows(doc, mode="doc", max_tok=MAX_TOK, ctx=CTX_LEFT):
    """-> list of (ctx_start, pred_start, end). Sentences [pred_start, end) are predicted; [ctx_start, pred_start) are
    read-only left context. mode='sent' = every sentence alone (the sentence-only / no-context baseline)."""
    n = len(doc["ids"])
    if mode == "sent":
        return [(i, i, i + 1) for i in range(n)]
    out, s = [], 0
    while s < n:
        c0 = max(0, s - ctx)
        used = sum(len(doc["ids"][i]) for i in range(c0, s))
        while used > max_tok // 2 and c0 < s:                     # context may use at most half the budget
            used -= len(doc["ids"][c0]); c0 += 1
        e = s
        while e < n and (e == s or used + len(doc["ids"][e]) <= max_tok):
            used += len(doc["ids"][e]); e += 1
        out.append((c0, s, e)); s = e
    return out

def collate(items):
    """items: list of (doc, ctx_start, pred_start, end) -> tensors.
    X (B,T) tokens | M (B,S,T) mean-pooling weights | last (B,S) index of each sentence's last token |
    nS (B,) sentences per window | Y (B,S) label ids, -100 for context sentences and padding."""
    seqs = [[t for i in range(c0, e) for t in d["ids"][i]] for d, c0, s, e in items]
    B, T, S = len(items), max(len(q) for q in seqs), max(e - c0 for _, c0, _, e in items)
    X = torch.zeros(B, T, dtype=torch.long)
    M = torch.zeros(B, S, T)
    last = torch.zeros(B, S, dtype=torch.long)
    Y = torch.full((B, S), -100, dtype=torch.long)
    nS = torch.tensor([e - c0 for _, c0, _, e in items])
    for b, (d, c0, s, e) in enumerate(items):
        X[b, :len(seqs[b])] = torch.tensor(seqs[b])
        pos = 0
        for j, i in enumerate(range(c0, e)):
            L = len(d["ids"][i])
            M[b, j, pos:pos + L] = 1.0 / L
            last[b, j] = pos + L - 1
            pos += L
            if i >= s and d["labels"] is not None:
                Y[b, j] = label2id[d["labels"][i]]
    return X, M, last, nS, Y

w = make_windows(train_docs[0]); print("windows of the first train doc (ctx_start, pred_start, end):", w[:5], "...", len(w), "windows")
print("windows per train set:", sum(len(make_windows(d)) for d in train_docs))

### 4.5 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
ids = [t for s in train_docs[0]["ids"][:40] for t in s][:512]
x = torch.tensor([ids]); _lm.eval()
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a train snippet:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3), "| tokens:", len(ids))
del _lm

## 5. Metric (macro P / R / F1 over all labels)

In [ ]:
def macro_scores(gold, pred, labels):
    """Macro precision / recall / F1 over ALL labels (a label with no gold and no predictions scores 0)."""
    P, R, F1, per = [], [], [], {}
    for l in labels:
        tp = sum(1 for g, p in zip(gold, pred) if g == l and p == l)
        npred = sum(1 for p in pred if p == l); ngold = sum(1 for g in gold if g == l)
        p = tp / npred if npred else 0.0; r = tp / ngold if ngold else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        P.append(p); R.append(r); F1.append(f); per[l] = (f, ngold)
    return dict(macro_f1=sum(F1) / len(F1), macro_p=sum(P) / len(P), macro_r=sum(R) / len(R), per_label=per)

## 6. Adaptation methods: LoRA, rsLoRA, LoRMA+, IA3 (prompt tuning lives in the model)
Adapters wrap the Linear layers (attention + FFN); the GRU weights themselves stay frozen under LoRA-style methods. LoRMA+ and the IA3 key/value scaling only touch the 4 attention blocks.

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)


class IA3Out(nn.Module):
    """IA3: learned vector rescales the layer OUTPUT (used for keys and values). Init = ones (identity)."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        return self.base(x) * self.l

class IA3In(nn.Module):
    """IA3 on the FFN: learned vector rescales the hidden activation that feeds the second FFN linear."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        return self.base(x * self.l)

def apply_ia3(blocks):
    """Attention blocks: scale keys, values and FFN activations. GRU blocks have no keys/values: FFN activations only."""
    for b in blocks:
        if hasattr(b, "k_proj"):
            b.k_proj = IA3Out(b.k_proj)
            b.v_proj = IA3Out(b.v_proj)
        b.ffn.net[2] = IA3In(b.ffn.net[2])

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

## 7. Model, `build_model`, distillation student

In [ ]:
class RRModel(nn.Module):
    """Phase-1 hybrid (8 GRU blocks + 4 attention blocks, all causal) -> per-sentence vector [mean of token states ; state of the sentence's last token]
    -> BiGRU across the sentences of the window (gives every sentence its left AND right neighbours) -> 13-way label.
    Because the window holds several consecutive sentences, the hybrid itself already sees the preceding text."""
    def __init__(self, lm, n_prompt=0, proj=512, rnn_hidden=256, dropout=0.1):
        super().__init__()
        self.lm, d = lm, lm.cfg.d_model
        self.n_prompt = n_prompt
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt else None     # prompt tuning
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d, proj), nn.GELU())
        self.rnn = nn.GRU(proj, rnn_hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * rnn_hidden, N_LAB)

    def head_params(self):
        ps = list(self.proj.parameters()) + list(self.rnn.parameters()) + list(self.out.parameters())
        return ps + ([self.prompt] if self.prompt is not None else [])

    def features(self, x):
        lm, B = self.lm, x.size(0)
        h = lm.tok_emb(x)
        if self.prompt is not None:
            h = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1), h], 1)
        # no positional embedding here: the hybrid gets position from its GRU blocks and RoPE
        for blk in lm.blocks:
            h = blk(h)
        return lm.ln_f(h)[:, self.n_prompt:]

    def forward(self, x, M, last, nS):
        h = self.features(x)                                              # (B, T, d)
        mean = M.to(h.dtype) @ h                                          # (B, S, d)
        lastv = h.gather(1, last.unsqueeze(-1).expand(-1, -1, h.size(-1)))
        z = self.proj(torch.cat([mean, lastv], -1))                       # (B, S, proj)
        packed = nn.utils.rnn.pack_padded_sequence(z, nS.cpu(), batch_first=True, enforce_sorted=False)
        z, _ = self.rnn(packed)
        z, _ = nn.utils.rnn.pad_packed_sequence(z, batch_first=True, total_length=M.size(1))
        return self.out(self.drop(z))                                     # (B, S, N_LAB)


def build_model(method, r=16, alpha=32, lorma_r=8, lorma_alpha=8, n_prompt=20):
    lm = load_lm()
    prompt = 0
    if method != "full" and method != "partial":
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":      apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "rslora":  apply_lora(lm.blocks, r, alpha, rs=True)
    elif method == "lorma":   apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)
    elif method == "ia3":     apply_ia3(lm.blocks)
    elif method == "prompt":  prompt = n_prompt
    elif method == "partial":                       # freeze embeddings + lower half of the blocks
        for p in lm.tok_emb.parameters(): p.requires_grad = False
        for b in lm.blocks[: len(lm.blocks) // 2]:
            for p in b.parameters(): p.requires_grad = False
    elif method != "full":
        raise ValueError(method)
    return RRModel(lm, n_prompt=prompt).to(dev)


def make_student(teacher):
    """Smaller student: from each [R R A] group keep [R A] (drop the 2nd GRU block) -> 8 of 12 blocks, shared embeddings,
    initialised from the teacher incl. its trained classification head. The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = [i for i in range(len(st.lm.blocks)) if i % 3 != 1]
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    for p in st.parameters(): p.requires_grad = True
    return st

## 8. Inference and evaluation

In [ ]:
@torch.no_grad()
def predict_logprobs(model, docs, mode="doc", bs=8):
    """-> {doc key: (n_sentences, N_LAB) log-probs}.  Every sentence is predicted exactly once (context is read-only)."""
    model.eval()
    items = [(d, c0, s, e) for d in docs for (c0, s, e) in make_windows(d, mode)]
    cost = [sum(len(d["ids"][k]) for k in range(c0, e)) for d, c0, s, e in items]
    order = sorted(range(len(items)), key=lambda i: cost[i])               # similar lengths together -> little padding
    out = {d["key"]: torch.zeros(len(d["ids"]), N_LAB) for d in docs}
    for i in range(0, len(order), bs):
        batch = [items[j] for j in order[i:i + bs]]
        X, M, last, nS, _ = collate(batch)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            lg = model(X.to(dev), M.to(dev), last.to(dev), nS)
        lp = F.log_softmax(lg.float(), -1).cpu()
        for b, (d, c0, s, e) in enumerate(batch):
            out[d["key"]][s:e] = lp[b, s - c0:e - c0]
    return out

def predict_labels(model, docs, mode="doc"):
    lp = predict_logprobs(model, docs, mode)
    return {d["key"]: [labels[i] for i in lp[d["key"]].argmax(-1).tolist()] for d in docs}

def evaluate(model, docs, mode="doc", verbose=True):
    pred = predict_labels(model, docs, mode)
    def flat(ds):
        return [l for d in ds for l in d["labels"]], [l for d in ds for l in pred[d["key"]]]
    res = macro_scores(*flat(docs), labels)
    res["by_domain"] = {dom: macro_scores(*flat([d for d in docs if d["dom"] == dom]), labels)["macro_f1"]
                        for dom in ("CL", "IT")}
    if verbose:
        print(f"   dev macro-F1 {res['macro_f1']:.4f} (P {res['macro_p']:.4f} R {res['macro_r']:.4f}) | "
              f"CL {res['by_domain']['CL']:.4f} IT {res['by_domain']['IT']:.4f}")
    return res

## 9. Training loop (all methods; best-epoch restore on dev)

In [ ]:
def train(method, train_docs, dev_docs, epochs=5, bs=4, accum=2, lr_bb=3e-5, lr_head=1e-3,
          teacher=None, kd_T=2.0, kd_alpha=0.5, cw_power=0.5, mode="doc", seed=0):
    """method: full | lora | rslora | lorma | ia3 | prompt | partial | kd.  mode: 'doc' (context) or 'sent' (sentence only).
    Returns (model restored to its best dev epoch, best dev macro-F1)."""
    random.seed(seed); torch.manual_seed(seed)
    if method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method)
    print(f"[{method}/{mode}] params total {n_params(model)/1e6:.1f}M | trainable {n_params(model, True)/1e6:.2f}M")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)

    items = [(d, c0, s, e) for d in train_docs for (c0, s, e) in make_windows(d, mode)]
    steps_total = epochs * math.ceil(math.ceil(len(items) / bs) / accum)
    warm = max(1, int(0.06 * steps_total))
    def lr_mult(s):
        return s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps_total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    cnt = torch.tensor([sum(l == lab for d in train_docs for l in d["labels"]) for lab in labels], dtype=torch.float) + 1
    cw = ((cnt.sum() / cnt / N_LAB) ** cw_power).to(dev) if cw_power else torch.ones(N_LAB, device=dev)
    cw = cw / cw.mean()                                          # class-balanced loss (cw_power=0 turns it off)

    best, best_state, step = -1.0, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        random.shuffle(items)
        batches = [items[i:i + bs] for i in range(0, len(items), bs)]
        for bi, batch in enumerate(batches):
            X, M, last, nS, Y = collate(batch)
            X, M, last, Y = X.to(dev), M.to(dev), last.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, M, last, nS)
            loss = F.cross_entropy(lg.float().reshape(-1, N_LAB), Y.reshape(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, M, last, nS)
                m = Y != -100
                kl = F.kl_div(F.log_softmax(lg.float()[m] / kd_T, -1), F.softmax(tl.float()[m] / kd_T, -1),
                              reduction="batchmean") * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kl
            scaler.scale(loss / accum).backward(); run += loss.item()
            if (bi + 1) % accum == 0 or bi + 1 == len(batches):
                step += 1
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * lr_mult(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        print(f"ep {ep+1}/{epochs} loss {run/max(1, len(batches)):.4f} ({time.time()-t0:.0f}s)")
        if dev_docs:
            sc = evaluate(model, dev_docs, mode)
            if sc["macro_f1"] > best:
                best = sc["macro_f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state); print(f"restored best epoch, dev macro-F1 {best:.4f}")
    return model, best

## 10. Submission writer
`prediction.json` goes at the **root** of the zip. Upload each zip to Codabench.

In [ ]:
def make_submission(model, docs, tag, mode="doc"):
    """prediction.json = {doc_id: [label per sentence]} for the CL and IT test docs, zipped with the json at the zip root."""
    pred = predict_labels(model, docs, mode)
    sub = {}
    for d in docs:
        assert len(pred[d["key"]]) == len(d["sents"])                   # exactly one label per sentence
        assert d["id"] not in sub, f"doc id {d['id']} appears in both domains - check how Codabench expects this"
        sub[d["id"]] = pred[d["key"]]
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False)
    name = f"{OUT}/submission_p3_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(name, zipfile.ZipFile(name).namelist(), "| docs:", len(sub), "| sentences:", sum(len(v) for v in sub.values()))
    return name

results = {}

## 11. Experiment 1 - full fine-tuning (also the KD teacher)
Everything trainable (backbone lr 3e-5, head lr 1e-3). Keep this model in memory for distillation.

In [ ]:
m_full, results["full"] = train("full", train_docs, dev_docs, lr_bb=3e-5)
torch.save(m_full.state_dict(), f"{OUT}/p3_hybrid_full.pt")
make_submission(m_full, test_docs, "hybrid_full")

## 12. Experiment 2 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer (attention and FFN).

In [ ]:
m_lora, results["lora"] = train("lora", train_docs, dev_docs, lr_bb=5e-4)
make_submission(m_lora, test_docs, "hybrid_lora")

## 13. Experiment 3 - rsLoRA
Same as LoRA but with the rank-stabilised scale alpha/sqrt(r).

In [ ]:
m_rs, results["rslora"] = train("rslora", train_docs, dev_docs, lr_bb=5e-4)
make_submission(m_rs, test_docs, "hybrid_rslora")

## 14. Experiment 4 - LoRMA+
Multiplicative low-rank adaptation of the query/value projections.

In [ ]:
m_lorma, results["lorma"] = train("lorma", train_docs, dev_docs, lr_bb=3e-4)
make_submission(m_lorma, test_docs, "hybrid_lorma")

## 15. Experiment 5 - IA3
Learned rescaling vectors on keys, values and FFN activations (very few parameters; usually wants a large lr).

In [ ]:
m_ia3, results["ia3"] = train("ia3", train_docs, dev_docs, lr_bb=3e-3)
make_submission(m_ia3, test_docs, "hybrid_ia3")

## 16. Experiment 6 - prompt tuning
20 learned soft-prompt vectors prepended to every window; the backbone stays frozen (the prompt trains at `lr_head`).

In [ ]:
m_prompt, results["prompt"] = train("prompt", train_docs, dev_docs)
make_submission(m_prompt, test_docs, "hybrid_prompt")

## 17. Experiment 7 - knowledge distillation
Teacher = experiment 1. Student = teacher with the 2nd GRU block of every [R R A] group removed (8 of 12 blocks), initialised from the teacher, trained on hard labels + temperature-scaled teacher logits. Compare its parameter count, speed and F1 with the teacher.

In [ ]:
m_kd, results["kd"] = train("kd", train_docs, dev_docs, teacher=m_full, lr_bb=5e-5)
make_submission(m_kd, test_docs, "hybrid_kd")

## 18. Context ablation - sentence-only
Same full fine-tune, but every sentence is classified alone (no neighbouring sentences). Compare with experiment 1 to answer *does context help?*

In [ ]:
m_sent, results["full_sentence_only"] = train("full", train_docs, dev_docs, lr_bb=3e-5, mode="sent")
make_submission(m_sent, test_docs, "hybrid_full_sentonly", mode="sent")

## 19. Summary and error-analysis starters

In [ ]:
print("dev macro-F1 (CL+IT):")
for k, v in results.items():
    print(f"  {k:20s} {v:.4f}")
sc = evaluate(m_full, dev_docs)
print("\nper-label F1 / dev support, full fine-tune (lowest first):")
for k, (f, n) in sorted(sc["per_label"].items(), key=lambda kv: kv[1][0]):
    print(f"  {k:24s} F1 {f:.3f}  n={n}")

# confusion between rhetorical roles (rows = gold, cols = predicted), for the report
pred = predict_labels(m_full, dev_docs)
conf = Counter((g, p) for d in dev_docs for g, p in zip(d["labels"], pred[d["key"]]) if g != p)
print("\nmost common confusions (gold -> predicted):")
for (g, p), c in conf.most_common(10):
    print(f"  {g:22s} -> {p:22s} {c}")

## 20. (Optional) retrain on train + dev
Once you know the best method and epoch count, retrain on all labelled data (no dev set, so no best-epoch restore) and submit again.
```python
m_final, _ = train("full", train_docs + dev_docs, None, epochs=<best epoch count>)
make_submission(m_final, test_docs, "decoder_final")
```